In [4]:

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json, time, zipfile

START = time.time()
PROJECT = Path("/kaggle/working/nd_iris_project")
PREP = PROJECT / "notebook02_preprocessing"
IMAGE_DIR = PREP / "images_224"
OUT = PROJECT / "notebook03_quality_assessment"
OUT.mkdir(parents=True, exist_ok=True)

assert IMAGE_DIR.exists(), f"Missing preprocessed images: {IMAGE_DIR}"
images = sorted(IMAGE_DIR.glob("*.png"))
assert images, "No preprocessed PNG images found."

print(f"Images to assess: {len(images):,}")

# Fast, interpretable image-quality indicators.
# Noise estimate is an exploratory high-frequency residual proxy, not calibrated noise.
def assess_image(path):
    with Image.open(path) as im:
        arr = np.asarray(im.convert("L"), dtype=np.float32)

    brightness = float(arr.mean())
    contrast = float(arr.std())
    dynamic_range = float(arr.max() - arr.min())

    # Gradient energy / sharpness proxy
    gx = np.diff(arr, axis=1)
    gy = np.diff(arr, axis=0)
    gradient_energy = float((np.mean(gx * gx) + np.mean(gy * gy)) / 2)

    # Variance of a simple Laplacian-like filter; higher often indicates more edges,
    # but can also respond to noise and texture.
    lap = (
        -4 * arr[1:-1, 1:-1]
        + arr[:-2, 1:-1] + arr[2:, 1:-1]
        + arr[1:-1, :-2] + arr[1:-1, 2:]
    )
    lap_var = float(np.var(lap))

    # Robust high-frequency residual proxy using a local four-neighbor average.
    center = arr[1:-1, 1:-1]
    neighbor_mean = (
        arr[:-2, 1:-1] + arr[2:, 1:-1]
        + arr[1:-1, :-2] + arr[1:-1, 2:]
    ) / 4
    residual = center - neighbor_mean
    noise_proxy = float(np.mean(np.abs(residual)))

    return {
        "filename": path.name,
        "brightness_mean": brightness,
        "contrast_std": contrast,
        "dynamic_range": dynamic_range,
        "gradient_energy": gradient_energy,
        "laplacian_variance": lap_var,
        "high_frequency_residual_proxy": noise_proxy,
    }

rows, errors = [], []
for p in tqdm(images, desc="Quality assessment"):
    try:
        rows.append(assess_image(p))
    except Exception as e:
        errors.append({"filename": p.name, "error": str(e)[:300]})

df = pd.DataFrame(rows)
df.to_csv(OUT / "ND_Iris_image_quality_metrics.csv", index=False)
pd.DataFrame(errors, columns=["filename", "error"]).to_csv(
    OUT / "ND_Iris_quality_assessment_errors.csv", index=False
)

# Descriptive statistics
metric_cols = [
    "brightness_mean", "contrast_std", "dynamic_range",
    "gradient_energy", "laplacian_variance",
    "high_frequency_residual_proxy"
]
stats = df[metric_cols].describe().T.reset_index(names="metric")
stats.to_csv(OUT / "ND_Iris_quality_statistics.csv", index=False)

# Exploratory flags based on dataset percentiles, not clinical/biometric standards.
flags = pd.DataFrame({"filename": df["filename"]})
flags["low_brightness_p10"] = df["brightness_mean"] < df["brightness_mean"].quantile(0.10)
flags["high_brightness_p90"] = df["brightness_mean"] > df["brightness_mean"].quantile(0.90)
flags["low_contrast_p10"] = df["contrast_std"] < df["contrast_std"].quantile(0.10)
flags["low_sharpness_p10"] = df["laplacian_variance"] < df["laplacian_variance"].quantile(0.10)
flags["high_residual_p90"] = (
    df["high_frequency_residual_proxy"]
    > df["high_frequency_residual_proxy"].quantile(0.90)
)
flags.to_csv(OUT / "ND_Iris_quality_flags.csv", index=False)

# Plot histograms
for metric in metric_cols:
    plt.figure(figsize=(7, 4))
    plt.hist(df[metric].dropna(), bins=40)
    plt.title(f"ND Iris: {metric.replace('_', ' ').title()}")
    plt.xlabel(metric.replace("_", " ").title())
    plt.ylabel("Image count")
    plt.tight_layout()
    plt.savefig(OUT / f"{metric}_distribution.png", dpi=130)
    plt.close()

# A small contact sheet of images with the lowest Laplacian variance.
# This is an exploratory review list, not a claim that these are unusable.
lowest = df.nsmallest(min(16, len(df)), "laplacian_variance")
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax in axes.flat:
    ax.axis("off")
for ax, (_, row) in zip(axes.flat, lowest.iterrows()):
    path = IMAGE_DIR / row["filename"]
    with Image.open(path) as im:
        ax.imshow(im.convert("L"), cmap="gray")
    ax.set_title(row["filename"][:20], fontsize=7)
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "lowest_sharpness_contact_sheet.png", dpi=140)
plt.close()

# Summary
summary = {
    "images_found": len(images),
    "images_assessed": len(df),
    "assessment_errors": len(errors),
    "metrics": {},
    "exploratory_flag_counts": {
        c: int(flags[c].sum()) for c in flags.columns if c != "filename"
    },
    "interpretation_notes": [
        "Brightness and contrast are descriptive pixel statistics.",
        "Laplacian variance and gradient energy are sharpness/edge proxies, not definitive quality scores.",
        "High-frequency residual is an exploratory noise proxy, not calibrated noise measurement.",
        "Percentile flags are relative to this dataset and are not validated rejection thresholds.",
        "Image quality does not establish identity-label correctness."
    ]
}
for metric in metric_cols:
    summary["metrics"][metric] = {
        "mean": float(df[metric].mean()),
        "std": float(df[metric].std()),
        "median": float(df[metric].median()),
        "min": float(df[metric].min()),
        "max": float(df[metric].max()),
    }

with open(OUT / "ND_Iris_quality_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# Preliminary conclusion report; a separate Markdown conclusion cell follows.
conclusion = f"""# Notebook 03 — Ocular Image Quality Assessment

- Images discovered: **{len(images):,}**
- Images assessed successfully: **{len(df):,}**
- Assessment errors: **{len(errors):,}**

Metrics include mean brightness, grayscale contrast, dynamic range, gradient energy,
Laplacian variance, and an exploratory high-frequency residual proxy. Distribution
plots, percentile-based review flags, and a contact sheet of low-sharpness candidates
were generated.

These indicators are exploratory. The percentile flags are relative to this dataset
and should not automatically be treated as image rejection criteria. Quality metrics
also do not validate identity labels.
"""
(OUT / "Notebook03_Conclusion.md").write_text(conclusion)

# ZIP all Notebook 03 outputs
ZIP_PATH = PROJECT / "ND_Iris_Notebook03_Results.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            z.write(p, arcname=f"notebook03_quality_assessment/{p.relative_to(OUT)}")

with zipfile.ZipFile(ZIP_PATH) as z:
    bad = z.testzip()
    archived_count = len(z.namelist())

print("\n" + "=" * 65)
print("NOTEBOOK 03 — QUALITY ASSESSMENT RESULTS")
print("=" * 65)
print(f"Images assessed: {len(df):,}/{len(images):,}")
print(f"Errors: {len(errors):,}")
print("\nMetric summary:")
print(stats.round(3).to_string(index=False))
print("\nExploratory flag counts:")
print(json.dumps(summary["exploratory_flag_counts"], indent=2))
print(f"\nConclusion report: {OUT / 'Notebook03_Conclusion.md'}")
print(f"Results ZIP: {ZIP_PATH}")
print(f"Archived result files: {archived_count}")
print(f"ZIP integrity: {'PASSED' if bad is None else 'FAILED: ' + str(bad)}")
print(f"Elapsed: {(time.time()-START)/60:.2f} minutes")
assert bad is None, "ZIP integrity check failed"


Images to assess: 6,233


Quality assessment:   0%|          | 0/6233 [00:00<?, ?it/s]


NOTEBOOK 03 — QUALITY ASSESSMENT RESULTS
Images assessed: 6,233/6,233
Errors: 0

Metric summary:
                       metric  count    mean     std     min     25%     50%     75%      max
              brightness_mean 6233.0 146.486  23.243  89.007 127.452 140.618 165.642  217.567
                 contrast_std 6233.0  61.586   8.497  29.859  55.992  61.090  66.415   92.549
                dynamic_range 6233.0 254.371   4.334 186.000 255.000 255.000 255.000  255.000
              gradient_energy 6233.0 167.315  64.986  50.656 106.127 166.056 221.066  425.853
           laplacian_variance 6233.0 720.668 239.021 181.910 527.893 708.626 883.348 1916.226
high_frequency_residual_proxy 6233.0   3.423   0.582   1.486   3.020   3.353   3.766    6.044

Exploratory flag counts:
{
  "low_brightness_p10": 624,
  "high_brightness_p90": 624,
  "low_contrast_p10": 624,
  "low_sharpness_p10": 624,
  "high_residual_p90": 624
}

Conclusion report: /kaggle/working/nd_iris_project/notebook03_quality_as

## Conclusion — Notebook 03: Ocular Image Quality Assessment

The ocular image quality assessment was successfully completed on all **6,233 preprocessed ND Iris images**, with zero assessment errors.

### Results

| Quality metric | Mean | Standard deviation |
|---|---:|---:|
| Brightness | 146.486 | 23.243 |
| Contrast | 61.586 | 8.497 |
| Dynamic range | 254.371 | 4.334 |
| Gradient energy | 167.315 | 64.986 |
| Laplacian variance | 720.668 | 239.021 |
| High-frequency residual proxy | 3.423 | 0.582 |

### Key Findings

- The mean brightness was **146.486**, with a standard deviation of 23.243.
- The mean contrast was **61.586**, indicating measurable variation in grayscale intensity across the images.
- The average dynamic range was **254.371** on the 8-bit grayscale scale.
- The mean Laplacian variance was **720.668** and the mean gradient energy was **167.315**. These provide exploratory indicators of edge detail and image sharpness.
- The high-frequency residual proxy averaged **3.423**. This is an exploratory indicator of local image variation, not a calibrated noise measurement.
- Each of the five percentile-based flags identified 624 images for further inspection.

### Generated Outputs

The notebook generated per-image quality metrics, descriptive statistics, exploratory quality flags, distribution plots, a low-sharpness contact sheet, a JSON summary, and `ND_Iris_Notebook03_Results.zip`.

**ZIP integrity:** Passed.

### Overall Conclusion

Notebook 03 establishes a quantitative baseline for examining image brightness, contrast, edge detail, and high-frequency variation in the ND Iris dataset. The calculated metrics can guide subsequent inspection and preprocessing decisions. However, the percentile flags are relative to this dataset and should not be interpreted as validated rejection criteria. Image quality assessment also does not establish the correctness of identity labels or guarantee biometric recognition performance.